In [1]:
!pip install -q transformers datasets peft accelerate safetensors


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import json

DATA_PATH = "/content/drive/Othercomputers/My Laptop/loan prediction via sml/training/lora_train.jsonl"

with open(DATA_PATH) as f:
    for i in range(3):
        print(json.loads(next(f)))


{'text': 'You are a conservative bank loan officer.\n\nApplicant Risk Profile:\n- Gender: Female\n- Marital Status: Unmarried\n- Dependents: Low\n- Employment Status: Employed\n- Coapplicant Income: No\n- Place of Living: Urban\n- Credit Score Quality: Good\n- EMI Repayment Capacity: High\n- Document Verification Status: Verified\n\nREASONING:\nApplicant has verified documents, good credit score, stable employment, low dependents, and strong EMI repayment capacity, indicating low credit risk.\n\nDECISION:\nApproved'}
{'text': 'You are a conservative bank loan officer.\n\nApplicant Risk Profile:\n- Gender: Male\n- Marital Status: Married\n- Dependents: Moderate\n- Employment Status: Employed\n- Coapplicant Income: Yes\n- Place of Living: Urban\n- Credit Score Quality: Good\n- EMI Repayment Capacity: High\n- Document Verification Status: Verified\n\nREASONING:\nVerified documentation combined with dual income, good credit history, and high repayment capacity strengthens loan eligibility.

In [5]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

In [6]:
MODEL_NAME = "/content/drive/Othercomputers/My Laptop/loan prediction via sml/models"
OUTPUT_DIR = "/content/drive/MyDrive/loan_prediction/lora_qwen_output"

In [19]:
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM = 16
EPOCHS = 7
LR = 8e-5

In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None
)

model.config.use_cache = False
model.train()

`torch_dtype` is deprecated! Use `dtype` instead!


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [11]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [12]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [13]:

print("Dataset size:", len(dataset))

Dataset size: 149


In [14]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, remove_columns=["text"])

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

In [20]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=True if DEVICE == "cuda" else False,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)

In [15]:
print("\n===== DRY FORWARD PASS =====")

sample = dataset[0]

inputs = {
    "input_ids": torch.tensor(sample["input_ids"]).unsqueeze(0).to(DEVICE),
    "attention_mask": torch.tensor(sample["attention_mask"]).unsqueeze(0).to(DEVICE),
    "labels": torch.tensor(sample["labels"]).unsqueeze(0).to(DEVICE),
}

model.train()
outputs = model(**inputs)
print("Initial loss:", outputs.loss.item())


===== DRY FORWARD PASS =====
Initial loss: 17.848102569580078


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [22]:
print("🚀 STARTING FINAL LoRA TRAINING (Qwen2.5)")
trainer.train()

🚀 STARTING FINAL LoRA TRAINING (Qwen2.5)


Step,Training Loss
10,0.486300
20,0.388200
30,0.353200
40,0.324500
50,0.309600
60,0.298600
70,0.291800


TrainOutput(global_step=70, training_loss=0.3503054073878697, metrics={'train_runtime': 275.3155, 'train_samples_per_second': 3.788, 'train_steps_per_second': 0.254, 'total_flos': 4212421012488192.0, 'train_loss': 0.3503054073878697, 'epoch': 7.0})

In [23]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ FINAL QWEN2.5 LoRA MODEL SAVED:", OUTPUT_DIR)

✅ FINAL QWEN2.5 LoRA MODEL SAVED: /content/drive/MyDrive/loan_prediction/lora_qwen_output
